# Tutorial: MLSecOps e Resposta a Incidentes

**Público:** ML engineers, SecOps e SRE.

**Objetivos:** correlacionar sinais, evitar “drift = ataque” e testar fallback/rollback.

## Roteiro

1. Configurar pipeline e versão segura.
2. Processar tráfego normal e suspeito.
3. Acionar rollback auditável.
4. Exercício: calibrar pesos e limiares.

> Pitfall: automatizar rollback crítico sem governança ou aprovação humana.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts" / "python"))
import numpy as np
from ml_security.incident_response import IncidentResponseManager
from ml_security.mlsecops_pipeline import MLSecOpsPipeline, RiskSignals

manager = IncidentResponseManager(model_version="2.3.0", safe_version="2.2.0")
pipeline = MLSecOpsPipeline(
    lambda values: (values.sum(axis=1) > 0).astype(int),
    feature_count=2,
    incident_manager=manager,
)

## Detectar, decidir e recuperar

Drift isolado permanece observação. Drift correlacionado a harvesting/extraction eleva o risco e muda a política.

In [2]:
normal = pipeline.process([0.1, 0.2], RiskSignals(drift_score=0.9))
abuse = pipeline.process(
    [0.1, 0.2],
    RiskSignals(drift_score=1.0, query_rate=1.0, extraction_suspicion=1.0),
)
rollback = manager.trigger_rollback("simulação controlada")
{"normal": normal, "abuse": abuse, "rollback": rollback}

{'normal': {'action': 'allow',
  'risk_score': 0.045000000000000005,
  'prediction': 1},
 'abuse': {'action': 'challenge',
  'risk_score': 0.55,
  'incident': {'incident_id': 'be8e60ff-8d4e-4f7a-b3e4-09b52655b4b9',
   'alert_type': 'ml_abuse',
   'severity': 'MEDIUM',
   'score': 0.55,
   'state': 'SUSPICIOUS',
   'action': 'challenge',
   'model_version': '2.3.0',
   'created_at': '2026-10-05T20:16:49.372304+00:00'}},
 'rollback': {'timestamp': '2026-10-05T20:16:49.372304+00:00',
  'action': 'rollback',
  'state': 'ROLLBACK',
  'model_version': '2.2.0',
  'details': {'reason': 'simulação controlada',
   'from': '2.3.0',
   'to': '2.2.0'}}}

## Exercício

Teste os estados NORMAL, SUSPICIOUS, DEGRADED, COMPROMISED e ROLLBACK. Confirme que toda ação aparece no audit log.

**Extensão:** conecte a saída da API endurecida ao RiskSignals.

In [3]:
{
    "current_model": manager.model_version,
    "state": manager.state.value,
    "incidents": [incident.to_dict() for incident in manager.incidents],
    "audit_events": len(manager.audit_log),
}

{'current_model': '2.2.0',
 'state': 'NORMAL',
 'incidents': [{'incident_id': 'be8e60ff-8d4e-4f7a-b3e4-09b52655b4b9',
   'alert_type': 'ml_abuse',
   'severity': 'MEDIUM',
   'score': 0.55,
   'state': 'SUSPICIOUS',
   'action': 'challenge',
   'model_version': '2.3.0',
   'created_at': '2026-10-05T20:16:49.372304+00:00'}],
 'audit_events': 2}